# 06 · 진짜 파이프라인 한 바퀴 돌려보기 (스모크 테스트)

> **공부 기록 노트북 6번.** 01~05 에서 *원리*를 배웠습니다. 이제
> `src/` 의 **진짜 학습 코드**를 돌려봅니다. 단, 끝까지(수 시간) 돌리지 않고
> **epochs=1 로 "연결만"** 확인합니다.

## 왜 이걸 따로 하나

`run_pipeline.ipynb` 는 분할(SAM2) + CVS 를 **끝까지** 학습합니다 — 무료
GPU 로 10시간 넘게 걸리고, 중간에 끊기면 부담입니다. 그 전에 알고 싶은 건
딱 하나:

> **"데이터 → 분할 학습 → CVS 학습 → 벤치마크"가 에러 없이 끝까지 이어지는가?**

이걸 확인하는 게 **스모크 테스트(smoke test)** 입니다. 전자제품을 처음 켤 때
*연기(smoke)가 나는지*만 보는 것에서 온 말 — 성능이 아니라 **"일단 돌아가는가"**
를 봅니다.

## 이 노트북의 선택

- **모델은 U-Net 베이스라인** (`model=unet_baseline`). SAM2 보다 가볍고 빨라
  연결 확인에 딱 맞습니다. (02 에서 구조를 배운 그 모델)
- **epochs=1** + 작은 이미지(256) → 한 단계가 짧게 끝납니다.
- 학습이 끝나면 **결과를 눈(예측 그림)과 숫자(벤치마크 표)로 확인**합니다.

⚠️ epochs=1 점수는 **당연히 낮습니다.** 여기서 보는 건 점수가 아니라
*"파이프라인이 끝까지 돌았다"* 입니다.

## 순서

1. 환경 + 데이터 준비
2. 분할 학습 (U-Net · 1 epoch)  → 결과 그림 확인
3. CVS 학습 (1 epoch, 위 분할 체크포인트 위에)
4. 벤치마크 → 결과 표 확인
5. 마무리

⚠️ **GPU 런타임 필요** — Runtime → Change runtime type → T4 GPU.
디스크 ~12GB (CholecSeg8k 3.1GB + Endoscapes 5.9GB). Drive 는 쓰지 않고
전부 `/content` 에서 돌립니다 (한 세션에 끝내는 전제).

## 0. 환경 준비

01~05 와 동일 — Colab 은 매번 빈 컴퓨터이므로 코드를 내려받고 라이브러리를
설치합니다. 여러 번 실행해도 안전합니다 (이미 있으면 최신 코드만 당겨옴).

> 코드는 GitHub `main` 에서 받습니다. 아직 머지 안 된 기능 브랜치를 쓰려면
> 아래 `-b main` 을 그 브랜치 이름으로 바꾸세요.

In [ ]:
%cd /content
import os
if not os.path.isdir("surgical-ai"):
    !git clone -b main https://github.com/duck-bin/surgical-ai.git
%cd surgical-ai
!git pull --ff-only
!bash scripts/colab_setup.sh

import torch
print("준비 완료 ·", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "없음 → Runtime > Change runtime type > GPU 로 바꾸세요")

## 1. 데이터 준비

epochs=1 이라도 **데이터는 진짜로** 필요합니다 (파이프라인이 데이터를 읽는
부분까지 확인해야 하니까). 두 데이터셋을 `/content` 로 받습니다 — Drive 안 씀.

- **CholecSeg8k** (~3.1GB) → 분할 학습용. HuggingFace 캐시(`./data/cholecseg8k`).
- **Endoscapes2023** (~5.9GB) → CVS 학습용. `/content/endoscapes2023`.

⚠️ 이 셀이 이 노트북에서 **가장 오래 걸립니다** (다운로드 ~20–40분, 네트워크
속도에 따라). 학습 자체보다 다운로드가 길 수 있어요. 한 번 받으면 같은
세션에선 캐시/`-c` 덕분에 다시 안 받습니다.

In [ ]:
# CholecSeg8k (~3.1 GB) -> 로컬 HuggingFace 캐시
!bash scripts/download_cholecseg8k.sh

# Endoscapes2023 (~5.9 GB) -> /content
!wget -q -c -O /content/endoscapes.zip https://s3.unistra.fr/camma_public/datasets/endoscapes/endoscapes.zip
!unzip -q -o /content/endoscapes.zip -d /content/endoscapes2023

### 데이터가 잘 읽히는지 먼저 확인

학습을 돌리기 전에, 두 로더가 프레임을 제대로 세는지 봅니다. **0 이 아닌
숫자**가 나오면 (CholecSeg8k 합계 8080) 데이터 계층은 정상입니다.

In [ ]:
from src.data.cholecseg8k import CholecSeg8kDataset
from src.data.endoscapes import Endoscapes2023Dataset

for split in ("train", "val", "test"):
    print(f"CholecSeg8k {split:5s}: {len(CholecSeg8kDataset(split=split))} frames")
for split in ("train", "val", "test"):
    ds = Endoscapes2023Dataset(root="/content/endoscapes2023", split=split)
    print(f"Endoscapes  {split:5s}: {len(ds)} CVS keyframes")

## 2. 분할 학습 — U-Net · 1 epoch

이제 **진짜 학습 진입점** `src/train/train_segmentation.py` 를 돌립니다.
02 의 손수 만든 미니 루프와 달리, 이건 검증·체크포인트 저장·스케줄러까지
포함한 *실제* 학습 코드(PyTorch Lightning)입니다.

스모크 테스트용 override 들:

| override | 의미 |
|---|---|
| `model=unet_baseline` | SAM2 대신 가벼운 U-Net 사용 |
| `epochs=1` | 한 바퀴만 (연결 확인이 목적) |
| `data.image_size=256` | 기본 1024 → 256 으로 줄여 대폭 단축 |
| `low_memory=false` | per-device 배치 4 (U-Net 은 T4 에 충분히 올라감) |

끝나면 `outputs/unet_baseline/best.ckpt` (와 `last.ckpt`) 가 생기고,
마지막에 `trainer.test` 가 **test_miou** 를 출력합니다 — 그 숫자가 셀
출력 맨 아래에 보이면 분할 단계가 끝까지 돈 것입니다.

⏱️ 대략 ~10–15분 (T4 기준). 진행 막대의 `loss` 가 줄고, `val_miou` 가
0 보다 큰 값으로 찍히면 정상입니다.

> 키워드: training entry point, Hydra override, checkpoint, smoke test

In [ ]:
!python -m src.train.train_segmentation \
    model=unet_baseline epochs=1 data.image_size=256 low_memory=false

## 3. 분할 결과 눈으로 확인

방금 학습한 체크포인트를 불러와, val 프레임 몇 장에 대해 예측을 그려
정답(ground truth)과 비교합니다. **1 epoch 이라 거칠지만**, 예측이 무작위가
아니라 정답의 *형태를 어느 정도 따라가면* 분할 학습이 제대로 연결된 것입니다.

(`load_segmentation_checkpoint` 는 Lightning 체크포인트에서 모델 가중치만
꺼내 주는 헬퍼입니다 — 벤치마크 코드가 쓰는 것과 같은 함수.)

In [ ]:
import torch
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

from src.train.train_segmentation import build_model
from src.eval.benchmark_runner import load_segmentation_checkpoint
from src.data.cholecseg8k import CholecSeg8kDataset, CLASS_NAMES
from src.data.transforms import build_eval_transforms

IMG = 256
device = "cuda" if torch.cuda.is_available() else "cpu"

model_cfg = OmegaConf.load("configs/model/unet_baseline.yaml")
model = build_model(model_cfg, pretrained=False)
model = load_segmentation_checkpoint(model, "outputs/unet_baseline/best.ckpt")
model = model.to(device).eval()

ds = CholecSeg8kDataset(split="val", image_size=IMG,
                        transform=build_eval_transforms(IMG))

N = 3
fig, ax = plt.subplots(N, 3, figsize=(12, 4 * N))
for i in range(N):
    sample = ds[i]
    image = sample["image"].unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(image).argmax(dim=1)[0].cpu()

    rgb = sample["image"].permute(1, 2, 0).numpy()
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)
    ax[i, 0].imshow(rgb);                                        ax[i, 0].set_title("input")
    ax[i, 1].imshow(sample["mask"], vmin=0, vmax=5, cmap="tab10"); ax[i, 1].set_title("ground truth")
    ax[i, 2].imshow(pred, vmin=0, vmax=5, cmap="tab10");          ax[i, 2].set_title("U-Net (1 epoch)")
    for a in ax[i]:
        a.axis("off")
plt.tight_layout()
plt.show()
print("6-class:", CLASS_NAMES)

## 4. CVS 학습 — 위 분할 모델 위에 · 1 epoch

이제 두 번째 단계. `src/train/train_cvs.py` 는 **방금 학습한 분할 모델을
freeze 해서** 6채널 마스크를 만들고, 거기에 RGB 3채널을 더한 9채널을
ViT 분류기에 먹여 CVS 3기준을 학습합니다 (05 에서 본 그 구조).

기본 설정은 분할 모델로 SAM2 를 기대하므로, **우리 U-Net 체크포인트를
가리키도록** override 합니다:

| override | 의미 |
|---|---|
| `data.root=/content/endoscapes2023` | 받아둔 Endoscapes 위치 |
| `segmentation.model_config=configs/model/unet_baseline.yaml` | freeze 할 분할 모델 종류 = U-Net |
| `segmentation.checkpoint=outputs/unet_baseline/best.ckpt` | 2번에서 만든 체크포인트 |
| `epochs=1`, `data.image_size=256` | 빠른 연결 확인 |

끝나면 `outputs/cvs_classifier/best.ckpt` 가 생기고, `trainer.test` 가
**test_map** (mAP) 와 **test_qwk** 를 출력합니다. 그 숫자가 보이면 CVS
단계까지 연결된 것입니다 (값 자체는 1 epoch 이라 낮음).

⏱️ 대략 ~10–20분 (T4 기준).

In [ ]:
!python -m src.train.train_cvs \
    epochs=1 data.root=/content/endoscapes2023 data.image_size=256 \
    segmentation.model_config=configs/model/unet_baseline.yaml \
    segmentation.checkpoint=outputs/unet_baseline/best.ckpt

## 5. 벤치마크 — 결과를 표로

마지막으로 `src/eval/benchmark_runner.py` 가 학습된 분할 체크포인트를
CholecSeg8k 테스트셋에서 평가해 **비교표**(mIoU, Cystic Duct Dice, 부트스트랩
95% 신뢰구간)를 `results/benchmark_table.md` 에 씁니다.

표에는 U-Net 과 SAM2 두 행이 있는데, 우리는 U-Net 만 학습했으므로 **U-Net 행만
숫자가 차고 SAM2 행은 `TBD`** (체크포인트 없음 → 건너뜀) 로 나옵니다 — 정상입니다.

> 평가도 256 해상도·배치 8 로 빠르게 돌립니다 (U-Net 을 256 에서 학습했으니
> 평가도 256 으로 일관).

In [ ]:
!python -m src.eval.benchmark_runner data.image_size=256 batch_size=8

from pathlib import Path
from IPython.display import Markdown, display

table = Path("results/benchmark_table.md").read_text()
print(table)
display(Markdown(table))

## 마무리 — 이 노트북 정리

### 무엇을 확인했나
이 노트북이 에러 없이 끝까지 돌았다면, **파이프라인 전체가 연결**된 것입니다:

```
데이터 다운로드  ✅
   → 분할 학습 (U-Net, 1 epoch)        → outputs/unet_baseline/best.ckpt  ✅
   → CVS 학습 (위 체크포인트 freeze)    → outputs/cvs_classifier/best.ckpt ✅
   → 벤치마크                          → results/benchmark_table.md       ✅
```

### 기억할 점
- **스모크 테스트 = "돌아가는가"** 확인. epochs=1 의 낮은 점수는 *정상*이며
  목표가 아니다. (좋은 점수는 충분한 epoch 학습에서 나온다)
- 진짜 학습 코드를 **override** 로 조절했다: `model=`, `epochs=`,
  `data.image_size=`, `segmentation.checkpoint=` 등. Hydra 설정의 어떤 값이든
  명령줄에서 바꿀 수 있다.
- CVS 단계는 **앞 단계의 분할 체크포인트**에 의존한다 — 2번 없이는 4번이 안 된다.

### 다음 단계 — 자신이 생겼다면
- **U-Net 을 제대로**: `epochs` 를 키워서 (예: `epochs=30`) 점수가 올라가는지.
- **주력 모델 SAM2**: `run_pipeline.ipynb` 로 SAM2+LoRA 를 끝까지 학습 →
  벤치마크 표에서 U-Net 과 비교 (02 에서 말한 "베이스라인 대비 이득").
- 데모: `app/gradio_demo.py` — 프레임을 올리면 분할 + CVS 점수를 보여준다.